# Experiment 1: Multi-source behavioural credit risk modelling

**Question:** Does historical credit and repayment behaviour improve the detection of high-risk applicants beyond
application information?

**Data:** Home Credit Default Risk, shared 100,000-applicant sample (stratified). Shared split: train 70% /
val 15% / test 15%. All choices are made on train or val. Test is only used for the final comparisons.

| Section | What it does |
|---|---|
| 0. Setup | Loads the shared data, splits and evaluation code |
| 1. Gradient boosting | Application only vs all sources, then tuned on validation |
| 2. Logistic regression | Interpretable baseline, C tuned on validation |
| 3. Ablation | One source at a time, same settings |
| 4. Richer features | Adds indicators of repayment severity, persistence and recent behaviour, then repeats the ablation |
| 5. Feature selection | Removes constant and correlated features, then refits |
| 6. Final model | The model we think is best, with a summary of all models |

Outputs (score files and tables) are written to `exp1_outputs/`. Run all cells from top to bottom.

**Compact output folder:** `best_model_scores.csv` (validation + test scores for the full 31-history-feature model), `sensitivity_scores.csv` (both demographic variants), `exp1_metrics.csv` (model performance) and `exp1_comparisons.csv` (paired tests), `features.parquet` (engineered applicant table), and `model_metadata.json` (feature definitions and settings). Intermediate model predictions stay in memory.

## 0. Setup
Shared data, shared split and the shared evaluation code (`eval_exp1.py`), used for every model below.

In [1]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "Exp_1/exp1.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the project Exp_1 folder from the current working directory.")
CLEAN = PROJECT_ROOT / "clean_data"
HERE = PROJECT_ROOT / "Exp_1" / "exp1_outputs"          # stable output path from root or Exp_1
HERE.mkdir(exist_ok=True)

splits = pd.read_csv(CLEAN / "splits.csv")
print(splits["split"].value_counts().to_string())

spec = importlib.util.spec_from_file_location("eval_exp1", PROJECT_ROOT / "Exp_1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

# Gradient boosting settings chosen by tuning in Section 1 (all sources, validation PR-AUC)
PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols, data):
    return pd.get_dummies(data[cols], dummy_na=True).astype("float32")
import json

def write_result_rows(rows, section, record_type, folder=HERE):
    """Intermediate results are displayed only; final CSVs are written in Section 6."""
    return




def update_metadata(values, folder=HERE):
    target = Path(folder) / "model_metadata.json"
    existing = json.loads(target.read_text()) if target.exists() else {}
    existing.update(values)
    target.write_text(json.dumps(existing, indent=2))


def evaluate_and_record(engine, predictions, labels=None, *, stage, folder=HERE, **kwargs):
    metrics, comparisons, evaluated = engine.compare_prediction_files(predictions, labels, **kwargs)
    write_result_rows(metrics, stage, "test_model", folder)
    write_result_rows(comparisons, stage, "paired_comparison", folder)
    return metrics, comparisons, evaluated


## 1. Gradient boosting: application only vs all sources, then tuned on validation

* **Features:** application table (120 features, gender and marital status excluded) and history aggregates
  from bureau, previous applications, instalments and credit cards (13 features).
* **Model:** HistGradientBoostingClassifier, default settings. Trained on train, scored on val and test.
* **Tuning:** 24 settings (learning rate, max leaf nodes, min samples per leaf, L2), chosen by validation PR-AUC.
  The number of trees is set by early stopping inside train.

In [2]:
EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]

app = pd.read_parquet(CLEAN / "application.parquet").drop(columns=["TARGET"])
app = app.merge(splits[["SK_ID_CURR", "TARGET", "split"]], on="SK_ID_CURR", how="inner")
app_cols = [c for c in app.columns if c not in EXCLUDE]
print(f"{len(app):,} applicants, {len(app_cols)} application features")

In [3]:
# Bureau: loans at other lenders
b = pd.read_parquet(CLEAN / "bureau.parquet")
bureau = b.groupby("SK_ID_CURR").agg(
    n_bureau=("SK_ID_BUREAU", "size"),
    bureau_debt=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_overdue=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    bureau_credit=("AMT_CREDIT_SUM", "sum"),
)
bureau["bureau_debt_ratio"] = bureau["bureau_debt"] / bureau["bureau_credit"].where(bureau["bureau_credit"] > 0)

# Previous Home Credit applications (duplicate submissions removed)
p = pd.read_parquet(CLEAN / "previous_application.parquet")
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["refused"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Refused").astype(int)
prev = p.groupby("SK_ID_CURR").agg(
    n_prev=("SK_ID_PREV", "size"),
    prev_refusal_rate=("refused", "mean"),
    prev_amt_mean=("AMT_APPLICATION", "mean"),
)

# Instalments: one row per instalment (split payments already merged in cleaning)
i = pd.read_parquet(CLEAN / "installments.parquet")
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]) > 0, np.nan)   # unpaid: not counted
i["under"] = (i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)).clip(lower=0)
inst = i.groupby("SK_ID_CURR").agg(
    n_inst=("SK_ID_PREV", "size"),
    late_rate=("late", "mean"),
    underpay_mean=("under", "mean"),
)

# Credit cards: utilisation
c = pd.read_parquet(CLEAN / "credit_card.parquet")
c["util"] = c["AMT_BALANCE"] / c["AMT_CREDIT_LIMIT_ACTUAL"].where(c["AMT_CREDIT_LIMIT_ACTUAL"] > 0)
cc = c.groupby("SK_ID_CURR").agg(cc_util_mean=("util", "mean"), cc_months=("MONTHS_BALANCE", "nunique"))

hist = bureau.join([prev, inst, cc], how="outer").reset_index()
hist_cols = [col for col in hist.columns if col != "SK_ID_CURR"]
print(f"History features: {len(hist_cols)} columns for {len(hist):,} applicants with any history")

In [4]:
feat = app.merge(hist, on="SK_ID_CURR", how="left")   # applicants with no history get NaN (model handles it)
assert feat["SK_ID_CURR"].is_unique

def fit_and_score(cols, name, seed=42):
    """Fit on train, score val + test, return SK_ID_CURR, score in memory."""
    X = pd.get_dummies(feat[cols], dummy_na=True).astype("float32")
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed)
    model.fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    print(f"{name}: {len(X.columns)} model inputs, {len(out):,} scored applicants (predictions kept in memory)")
    return out

p_app = fit_and_score(app_cols, "exp1_app_only")
p_all = fit_and_score(app_cols + hist_cols, "exp1_all_sources")

In [5]:
spec = importlib.util.spec_from_file_location("eval_exp1", PROJECT_ROOT / "Exp_1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

table, comps, _ = evaluate_and_record(ev, [p_app, p_all], ["Application only", "All sources"], n_boot=1000, stage="initial_13_history")
print("Test-set metrics (value with 95% CI); threshold chosen on validation")
display(ev.pretty_metrics(table))
print("Paired comparison vs the application-only baseline")
display(ev.pretty_comparisons(comps))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application only,0.132,0.738 (0.724-0.753),0.211 (0.191-0.230),0.208 (0.194-0.223),0.443 (0.415-0.469),0.283 (0.266-0.301),0.241 (0.219-0.263),0.298 (0.275-0.322)
All sources,0.153,0.748 (0.733-0.763),0.234 (0.214-0.256),0.231 (0.213-0.250),0.387 (0.359-0.415),0.289 (0.270-0.310),0.257 (0.235-0.279),0.319 (0.295-0.341)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources,Application only,roc_auc,+0.0095 (+0.0033 to +0.0151),0.002,significantly better
1,All sources,Application only,pr_auc,+0.0234 (+0.0116 to +0.0351),0,significantly better


### 1b. Tuning on validation

In [6]:
import itertools
from sklearn.metrics import average_precision_score, roc_auc_score

GRID = {
    "learning_rate": [0.03, 0.1],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [20, 100],
    "l2_regularization": [0.0, 1.0],
}
COMBOS = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]   # 24 settings
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

def tune(cols, name):
    X = model_inputs(cols)
    tr, va = feat["split"] == "train", feat["split"] == "val"
    y_va = feat.loc[va, "TARGET"]
    rows = []
    for params in COMBOS:
        m = HistGradientBoostingClassifier(**params, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
        s = m.predict_proba(X[va])[:, 1]
        rows.append({**params, "trees": m.n_iter_,
                     "val_roc_auc": roc_auc_score(y_va, s), "val_pr_auc": average_precision_score(y_va, s)})
    res = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
    print(f"{name}: top 5 of {len(COMBOS)} settings by validation PR-AUC")
    display(res.head(5).round(4))
    top = res.iloc[0]
    best = {"learning_rate": float(top["learning_rate"]), "max_leaf_nodes": int(top["max_leaf_nodes"]),
            "min_samples_leaf": int(top["min_samples_leaf"]), "l2_regularization": float(top["l2_regularization"])}
    print("Chosen:", best)
    return best, res

best_app, res_app = tune(app_cols, "Application only")
best_all, res_all = tune(app_cols + hist_cols, "All sources")

,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,trees,val_roc_auc,val_pr_auc
0,0.10,31,100,0.0,94,0.7317,0.2238
1,0.03,15,100,1.0,278,0.7367,0.2237
2,0.03,15,100,0.0,302,0.7364,0.2236
3,0.03,31,100,0.0,209,0.7369,0.2235
4,0.03,31,100,1.0,302,0.7389,0.2226


,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,trees,val_roc_auc,val_pr_auc
0,0.03,15,100,1.0,322,0.7531,0.2383
1,0.03,31,20,1.0,324,0.7554,0.2358
2,0.03,15,100,0.0,364,0.7536,0.2352
3,0.10,15,100,1.0,162,0.7501,0.2350
4,0.03,15,20,1.0,468,0.7528,0.2339


In [7]:
def fit_tuned(cols, params, name, seed=42):
    """Refit the chosen settings on train only, score val + test, return predictions in memory."""
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**params, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    print(f"{name}: trees used = {model.n_iter_}, predictions kept in memory")
    return out

p_app_t = fit_tuned(app_cols, best_app, "exp1_app_only_tuned")
p_all_t = fit_tuned(app_cols + hist_cols, best_all, "exp1_all_sources_tuned")

In [8]:
print("Application only: default vs tuned")
t1, c1, _ = evaluate_and_record(ev, [p_app, p_app_t], ["Application only (default)", "Application only (tuned)"], n_boot=1000, stage="tuned_13_history")
display(ev.pretty_metrics(t1))
display(ev.pretty_comparisons(c1))

print("All sources: default vs tuned")
t2, c2, _ = evaluate_and_record(ev, [p_all, p_all_t], ["All sources (default)", "All sources (tuned)"], n_boot=1000, stage="tuned_13_history")
display(ev.pretty_metrics(t2))
display(ev.pretty_comparisons(c2))

print("Tuned models: application only vs all sources")
t3, c3, _ = evaluate_and_record(ev, [p_app_t, p_all_t], ["Application only (tuned)", "All sources (tuned)"], n_boot=1000, stage="tuned_13_history")
display(ev.pretty_metrics(t3))
display(ev.pretty_comparisons(c3))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application only (default),0.132,0.738 (0.724-0.753),0.211 (0.191-0.230),0.208 (0.194-0.223),0.443 (0.415-0.469),0.283 (0.266-0.301),0.241 (0.219-0.263),0.298 (0.275-0.322)
Application only (tuned),0.146,0.736 (0.721-0.750),0.211 (0.193-0.232),0.216 (0.200-0.232),0.402 (0.374-0.429),0.281 (0.262-0.299),0.244 (0.223-0.265),0.302 (0.279-0.324)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,Application only (tuned),Application only (default),roc_auc,-0.0025 (-0.0068 to +0.0015),0.24,no significant difference
1,Application only (tuned),Application only (default),pr_auc,+0.0003 (-0.0073 to +0.0071),1,no significant difference


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
All sources (default),0.153,0.748 (0.733-0.763),0.234 (0.214-0.256),0.231 (0.213-0.250),0.387 (0.359-0.415),0.289 (0.270-0.310),0.257 (0.235-0.279),0.319 (0.295-0.341)
All sources (tuned),0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources (tuned),All sources (default),roc_auc,+0.0052 (+0.0025 to +0.0081),0.00031,significantly better
1,All sources (tuned),All sources (default),pr_auc,+0.0028 (-0.0039 to +0.0092),0.41,no significant difference


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application only (tuned),0.146,0.736 (0.721-0.750),0.211 (0.193-0.232),0.216 (0.200-0.232),0.402 (0.374-0.429),0.281 (0.262-0.299),0.244 (0.223-0.265),0.302 (0.279-0.324)
All sources (tuned),0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources (tuned),Application only (tuned),roc_auc,+0.0172 (+0.0107 to +0.0232),1.5e-07,significantly better
1,All sources (tuned),Application only (tuned),pr_auc,+0.0259 (+0.0140 to +0.0363),0,significantly better


## 2. Logistic regression baseline (C tuned on validation)

The interpretable baseline the proposal asks for. Median imputation and standard scaling are fitted on train
only, inside a pipeline. The regularisation strength C is chosen by validation PR-AUC.

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

C_GRID = [0.01, 0.1, 1.0, 10.0]

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

def make_lr(C):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, max_iter=2000))

def tune_lr(cols, name):
    X = model_inputs(cols)
    tr, va = feat["split"] == "train", feat["split"] == "val"
    y_va = feat.loc[va, "TARGET"]
    rows = []
    for C in C_GRID:
        m = make_lr(C).fit(X[tr], feat.loc[tr, "TARGET"])
        s = m.predict_proba(X[va])[:, 1]
        rows.append({"C": C, "val_roc_auc": roc_auc_score(y_va, s), "val_pr_auc": average_precision_score(y_va, s)})
    res = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
    print(f"{name}: C tuned on validation PR-AUC")
    display(res.round(4))
    best = float(res.iloc[0]["C"])
    print("Chosen C:", best)
    return best

C_app = tune_lr(app_cols, "LR application only")
C_all = tune_lr(app_cols + hist_cols, "LR all sources")

,C,val_roc_auc,val_pr_auc
0,1.00,0.7243,0.1971
1,10.00,0.7242,0.1970
2,0.10,0.7245,0.1969
3,0.01,0.7242,0.1962


,C,val_roc_auc,val_pr_auc
0,0.10,0.7396,0.2162
1,10.00,0.7394,0.2160
2,0.01,0.7396,0.2158
3,1.00,0.7393,0.2158


In [10]:
def fit_lr_and_save(cols, C, name):
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = make_lr(C).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    print(f"{name}: C={C}, {len(out):,} scored applicants (predictions kept in memory)")
    return out

p_lr_app = fit_lr_and_save(app_cols, C_app, "exp1_lr_app_only")
p_lr_all = fit_lr_and_save(app_cols + hist_cols, C_all, "exp1_lr_all_sources")

In [11]:
spec = importlib.util.spec_from_file_location("eval_exp1", PROJECT_ROOT / "Exp_1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

print("Logistic regression: application only vs all sources")
t1, c1, _ = evaluate_and_record(ev, [p_lr_app, p_lr_all], ["LR application only", "LR all sources"], n_boot=1000, stage="logistic_13_history")
display(ev.pretty_metrics(t1))
display(ev.pretty_comparisons(c1))

print("All sources: logistic regression vs gradient boosting (tuned, from notebook 01)")
t2, c2, _ = evaluate_and_record(ev, [p_lr_all, p_all_t],
                                        ["LR all sources", "GBM all sources (tuned)"], n_boot=1000, stage="logistic_13_history")
display(ev.pretty_metrics(t2))
display(ev.pretty_comparisons(c2))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
LR application only,0.151,0.733 (0.717-0.747),0.205 (0.186-0.225),0.213 (0.196-0.231),0.365 (0.338-0.392),0.269 (0.250-0.288),0.239 (0.217-0.261),0.296 (0.274-0.320)
LR all sources,0.156,0.741 (0.726-0.754),0.211 (0.192-0.232),0.228 (0.211-0.245),0.373 (0.347-0.400),0.283 (0.264-0.301),0.247 (0.226-0.269),0.306 (0.283-0.328)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,LR all sources,LR application only,roc_auc,+0.0080 (+0.0032 to +0.0125),0.0013,significantly better
1,LR all sources,LR application only,pr_auc,+0.0061 (-0.0010 to +0.0136),0.1,no significant difference


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
LR all sources,0.156,0.741 (0.726-0.754),0.211 (0.192-0.232),0.228 (0.211-0.245),0.373 (0.347-0.400),0.283 (0.264-0.301),0.247 (0.226-0.269),0.306 (0.283-0.328)
GBM all sources (tuned),0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,GBM all sources (tuned),LR all sources,roc_auc,+0.0125 (+0.0054 to +0.0198),0.00088,significantly better
1,GBM all sources (tuned),LR all sources,pr_auc,+0.0260 (+0.0149 to +0.0372),0,significantly better


## 3. Ablation: one source at a time (same settings)

Adds one source group at a time to the application features, using the Section 1 gradient boosting settings for
every row. The difference between rows shows what each source adds, not what tuning adds.

In [12]:
feat = app.merge(hist, on="SK_ID_CURR", how="left")
assert feat["SK_ID_CURR"].is_unique

BUREAU = ["n_bureau", "bureau_debt", "bureau_overdue", "bureau_credit", "bureau_debt_ratio"]
PREVIOUS = ["n_prev", "prev_refusal_rate", "prev_amt_mean"]
REPAYMENT = ["n_inst", "late_rate", "underpay_mean", "cc_util_mean", "cc_months"]
assert set(BUREAU + PREVIOUS + REPAYMENT) == set(hist_cols)

In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier

PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

In [14]:
GROUPS = {
    "Application": app_cols,
    "+ Bureau": app_cols + BUREAU,
    "+ Previous": app_cols + PREVIOUS,
    "+ Repayment": app_cols + REPAYMENT,
    "All sources": app_cols + hist_cols,
}
FILES = {}
for name, cols in GROUPS.items():
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    FILES[name] = out
    print(f"{name:<12} inputs={X.shape[1]:>4}  trees={model.n_iter_:>4} (predictions kept in memory)")

In [15]:
spec = importlib.util.spec_from_file_location("eval_exp1", PROJECT_ROOT / "Exp_1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(GROUPS)
table, comps, _ = evaluate_and_record(ev, [FILES[l] for l in labels], labels, n_boot=1000, stage="ablation_13_history")
print("Test-set metrics (value with 95% CI); threshold chosen on validation")
display(ev.pretty_metrics(table))
print("Each source group vs the application-only baseline")
display(ev.pretty_comparisons(comps))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application,0.156,0.741 (0.726-0.754),0.215 (0.197-0.236),0.228 (0.211-0.246),0.377 (0.350-0.403),0.284 (0.265-0.303),0.246 (0.224-0.268),0.305 (0.281-0.328)
+ Bureau,0.166,0.743 (0.728-0.757),0.221 (0.201-0.242),0.235 (0.217-0.254),0.357 (0.331-0.382),0.283 (0.264-0.302),0.250 (0.231-0.273),0.310 (0.288-0.332)
+ Previous,0.153,0.743 (0.728-0.757),0.221 (0.201-0.243),0.225 (0.208-0.243),0.384 (0.357-0.411),0.283 (0.265-0.303),0.245 (0.224-0.266),0.304 (0.281-0.327)
+ Repayment,0.153,0.751 (0.737-0.765),0.232 (0.211-0.254),0.228 (0.212-0.246),0.391 (0.366-0.418),0.288 (0.270-0.308),0.252 (0.231-0.275),0.312 (0.290-0.336)
All sources,0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,+ Bureau,Application,roc_auc,+0.0023 (-0.0006 to +0.0053),0.13,no significant difference
1,+ Bureau,Application,pr_auc,+0.0052 (-0.0010 to +0.0108),0.1,no significant difference
2,+ Previous,Application,roc_auc,+0.0024 (-0.0007 to +0.0052),0.13,no significant difference
3,+ Previous,Application,pr_auc,+0.0060 (-0.0002 to +0.0110),0.062,no significant difference
4,+ Repayment,Application,roc_auc,+0.0109 (+0.0064 to +0.0153),7.5e-07,significantly better
5,+ Repayment,Application,pr_auc,+0.0164 (+0.0076 to +0.0242),0,significantly better
6,All sources,Application,roc_auc,+0.0125 (+0.0074 to +0.0173),5e-06,significantly better
7,All sources,Application,pr_auc,+0.0215 (+0.0106 to +0.0309),0,significantly better


## 4. Richer behavioural features

The initial 13 historical aggregates summarise debt, application outcomes and average repayment behaviour. They do not fully describe **how severe or persistent repayment problems are**, whether recent behaviour differs from the longer-term pattern, or how credit at other lenders performed month by month. We therefore add 18 indicators covering these aspects and repeat the source ablation.

The resulting specification contains **31 historical features** and 120 application features: 151 columns before categorical encoding and 281 model inputs in the current run, excluding gender and marital status.

| Source | Additional features | Information captured |
|---|---:|---|
| POS/cash | 3 | Worst days past due, overdue records and remaining instalments |
| Credit card | 4 | Months below minimum payment, overdue months, minimum-payment ratio and recent utilisation relative to the overall mean |
| Instalments | 4 | Worst and average days late, underpayment frequency and total paid relative to total due |
| Bureau | 2 | Worst days overdue and active-loan count |
| Previous applications | 2 | Approval rate and applications in the last year |
| Bureau balance | 3 | Worst monthly delinquency status, number of delinquent loan-month records and their share |

**Why include bureau balance?** The bureau table summarises individual loans, whereas bureau balance provides their monthly status histories. `bb_max_dpd` captures severity, `bb_months_with_dpd` captures the volume of recorded delinquency, and `bb_ever_late_share` expresses it relative to the number of records. Together they complement outstanding debt and current overdue amounts.

`STATUS_DPD` is an ordinal delinquency bucket, not an exact number of overdue days. The maximum ignores missing status values. For the count and share, missing status does not trigger the delinquency flag; the share uses all loan-month records as its denominator, including records with unknown status. Counts refer to **loan-month records**, not distinct calendar months. Applicants without bureau-balance records retain NaN after joining. These features enter **+ Bureau** and **All sources** only.

All five groups use the previously selected settings: learning rate 0.03, 15 leaves, minimum leaf size 100, L2 regularisation 1.0 and seed 42. Holding these settings fixed allows the comparison to focus on the additional features. Sections 1–3 retain the initial feature experiments; this section evaluates the richer specification. Section 7 separately examines the choice to include gender and marital status.

Only best-model and sensitivity scores are exported. Model performance is saved in `exp1_metrics.csv`, and paired tests are saved in `exp1_comparisons.csv`; the existing scores supplied for investigation remain unchanged.

In [4]:
feat_old = app.merge(hist, on="SK_ID_CURR", how="left")
BUREAU = ["n_bureau", "bureau_debt", "bureau_overdue", "bureau_credit", "bureau_debt_ratio"]
PREVIOUS = ["n_prev", "prev_refusal_rate", "prev_amt_mean"]
REPAYMENT = ["n_inst", "late_rate", "underpay_mean", "cc_util_mean", "cc_months"]
assert set(BUREAU + PREVIOUS + REPAYMENT) == set(hist_cols)

In [5]:
# POS / cash loans
q = pd.read_parquet(CLEAN / "pos_cash.parquet", columns=["SK_ID_CURR", "SK_DPD", "CNT_INSTALMENT_FUTURE"])
q["dpd_flag"] = (q["SK_DPD"] > 0).astype("int32")
pos = q.groupby("SK_ID_CURR").agg(
    pos_max_dpd=("SK_DPD", "max"),
    pos_months_dpd=("dpd_flag", "sum"),
    pos_remaining_inst=("CNT_INSTALMENT_FUTURE", "mean"),
)

# Credit card, month by month (one row per applicant-month, summed over contracts)
c = pd.read_parquet(CLEAN / "credit_card.parquet", columns=[
    "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_TOTAL_CURRENT", "SK_DPD"])
c["below_min"] = ((c["AMT_INST_MIN_REGULARITY"] > 0) &
                  (c["AMT_PAYMENT_TOTAL_CURRENT"].fillna(0) < c["AMT_INST_MIN_REGULARITY"])).astype("int32")
c["dpd_flag"] = (c["SK_DPD"] > 0).astype("int32")
c["min_ratio"] = c["AMT_PAYMENT_TOTAL_CURRENT"] / c["AMT_INST_MIN_REGULARITY"].where(c["AMT_INST_MIN_REGULARITY"] > 0)
m = c.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False).agg(
    bal=("AMT_BALANCE", "sum"), lim=("AMT_CREDIT_LIMIT_ACTUAL", "sum"),
    below=("below_min", "max"), dpd=("dpd_flag", "max"), min_ratio=("min_ratio", "mean"))
m["util"] = m["bal"] / m["lim"].where(m["lim"] > 0)
recent = m[m["MONTHS_BALANCE"] >= -3].groupby("SK_ID_CURR")["util"].mean()
card = m.groupby("SK_ID_CURR").agg(
    cc_months_below_min=("below", "sum"),
    cc_months_dpd=("dpd", "sum"),
    cc_min_ratio_mean=("min_ratio", "mean"),
    cc_util_overall=("util", "mean"),
)
card["cc_balance_trend"] = recent.reindex(card.index) - card["cc_util_overall"]
card = card.drop(columns="cc_util_overall")

# Instalments
i = pd.read_parquet(CLEAN / "installments.parquet", columns=[
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"])
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["days_late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]).clip(lower=0), np.nan)
i["under_flag"] = ((i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)) > 1.0).astype("int32")
i["paid_amt"] = i["AMT_PAYMENT"].fillna(0)
inst_new = i.groupby("SK_ID_CURR").agg(
    inst_max_days_late=("days_late", "max"),
    inst_mean_days_late=("days_late", "mean"),
    inst_underpay_freq=("under_flag", "mean"),
    paid_sum=("paid_amt", "sum"),
    due_sum=("AMT_INSTALMENT", "sum"),
)
inst_new["inst_paid_to_due"] = inst_new["paid_sum"] / inst_new["due_sum"].where(inst_new["due_sum"] > 0)
inst_new = inst_new.drop(columns=["paid_sum", "due_sum"])

# Bureau (loan-level)
b = pd.read_parquet(CLEAN / "bureau.parquet", columns=["SK_ID_CURR", "CREDIT_DAY_OVERDUE", "CREDIT_ACTIVE"])
bureau_new = b.groupby("SK_ID_CURR").agg(
    bureau_max_days_overdue=("CREDIT_DAY_OVERDUE", "max"),
    bureau_active_count=("CREDIT_ACTIVE", lambda s: (s.astype(str) == "Active").sum()),
)

# Bureau-balance summaries of monthly delinquency at external lenders
bb_flat = pd.read_parquet(CLEAN / "bureau_balance.parquet", columns=["SK_ID_CURR", "STATUS_DPD"])
bb_flat["dpd_flag"] = (bb_flat["STATUS_DPD"].fillna(0) > 0).astype("int32")
bb_agg = bb_flat.groupby("SK_ID_CURR").agg(
    bb_max_dpd=("STATUS_DPD", "max"),
    bb_months_with_dpd=("dpd_flag", "sum"),
    bb_ever_late_share=("dpd_flag", "mean"),
)
print(f"Bureau balance flat: {len(bb_agg):,} applicants")

# Previous applications
p = pd.read_parquet(CLEAN / "previous_application.parquet", columns=[
    "SK_ID_CURR", "NAME_CONTRACT_STATUS", "DAYS_DECISION", "IS_DUPLICATE_SUBMISSION"])
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["approved"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Approved").astype("int32")
p["last_year"] = (p["DAYS_DECISION"] >= -365).astype("int32")
prev_new = p.groupby("SK_ID_CURR").agg(prev_approval_rate=("approved", "mean"), prev_apps_last_year=("last_year", "sum"))

new_hist = pos.join([card, inst_new, bureau_new, bb_agg, prev_new], how="outer").reset_index()
NEW_POS = ["pos_max_dpd", "pos_months_dpd", "pos_remaining_inst"]
NEW_CARD = ["cc_months_below_min", "cc_months_dpd", "cc_min_ratio_mean", "cc_balance_trend"]
NEW_INST = ["inst_max_days_late", "inst_mean_days_late", "inst_underpay_freq", "inst_paid_to_due"]
NEW_BUREAU = ["bureau_max_days_overdue", "bureau_active_count"]
NEW_BB = ["bb_max_dpd", "bb_months_with_dpd", "bb_ever_late_share"]
NEW_PREV = ["prev_approval_rate", "prev_apps_last_year"]
assert set(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV) == set(new_hist.columns) - {"SK_ID_CURR"}
print(f"New features: {len(new_hist.columns) - 1} columns for {len(new_hist):,} applicants")
display(new_hist.describe().T[["count", "mean", "min", "max"]].round(3))

,count,mean,min,max
SK_ID_CURR,99290.0,277928.762,100011.00,456248.000
pos_max_dpd,94164.0,15.262,0.00,4231.000
pos_months_dpd,94164.0,0.885,0.00,120.000
pos_remaining_inst,94155.0,9.215,0.00,59.500
cc_months_below_min,28368.0,2.096,0.00,70.000
cc_months_dpd,28368.0,1.546,0.00,94.000
cc_min_ratio_mean,19351.0,34.067,0.00,471288.531
cc_balance_trend,21320.0,0.007,-0.97,2.767
inst_max_days_late,94873.0,18.639,0.00,2852.000
inst_mean_days_late,94873.0,1.282,0.00,2813.000


In [6]:
feat = feat_old.merge(new_hist, on="SK_ID_CURR", how="left")
assert feat["SK_ID_CURR"].is_unique

GROUPS = {
    "Application": app_cols,
    "+ Bureau": app_cols + BUREAU + NEW_BUREAU + NEW_BB,
    "+ Previous": app_cols + PREVIOUS + NEW_PREV,
    "+ Repayment": app_cols + REPAYMENT + NEW_POS + NEW_CARD + NEW_INST,
    "All sources": app_cols + hist_cols + NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV,
}
# 31 historical features spanning loan summaries and monthly repayment behaviour.
assert len(hist_cols) == 13
assert len(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV) == 18
assert len(GROUPS["All sources"]) == len(app_cols) + 31
assert len(GROUPS["All sources"]) == len(set(GROUPS["All sources"]))


In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

FILES = {}
for name, cols in GROUPS.items():
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    FILES[name] = out
    print(f"{name:<12} inputs={X.shape[1]:>4}  trees={model.n_iter_:>4} (predictions kept in memory)")
rich_predictions = {name: df.copy() for name, df in FILES.items()}


In [8]:
spec = importlib.util.spec_from_file_location("eval_exp1", PROJECT_ROOT / "Exp_1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(GROUPS)
table, comps, _ = evaluate_and_record(ev, [FILES[l] for l in labels], labels, n_boot=1000, stage="ablation_31_history")
display(ev.pretty_metrics(table))
display(ev.pretty_comparisons(comps))

,threshold,pr_auc,roc_auc,precision,recall,f1
model,,,,,,
Application,0.166,0.215 (0.196-0.237),0.741 (0.726-0.755),0.235 (0.217-0.254),0.350 (0.324-0.376),0.281 (0.260-0.301)
+ Bureau,0.157,0.223 (0.204-0.245),0.743 (0.728-0.757),0.229 (0.212-0.246),0.381 (0.355-0.406),0.286 (0.267-0.305)
+ Previous,0.150,0.222 (0.202-0.243),0.743 (0.728-0.757),0.221 (0.204-0.238),0.391 (0.364-0.419),0.282 (0.264-0.302)
+ Repayment,0.145,0.242 (0.221-0.265),0.757 (0.742-0.771),0.230 (0.213-0.247),0.433 (0.406-0.459),0.300 (0.281-0.319)
All sources,0.144,0.250 (0.228-0.273),0.758 (0.743-0.772),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321)


,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,+ Bureau,Application,pr_auc,+0.0080 (+0.0013 to +0.0139),0.026,0.1,no significant difference
1,+ Bureau,Application,roc_auc,+0.0021 (-0.0011 to +0.0052),0.23,0.45,no significant difference
2,+ Previous,Application,pr_auc,+0.0067 (+0.0005 to +0.0125),0.038,0.11,no significant difference
3,+ Previous,Application,roc_auc,+0.0019 (-0.0010 to +0.0048),0.23,0.45,no significant difference
4,+ Repayment,Application,pr_auc,+0.0271 (+0.0170 to +0.0361),0,0,significantly better
5,+ Repayment,Application,roc_auc,+0.0155 (+0.0104 to +0.0211),9e-09,5.4e-08,significantly better
6,All sources,Application,pr_auc,+0.0351 (+0.0233 to +0.0455),0,0,significantly better
7,All sources,Application,roc_auc,+0.0165 (+0.0106 to +0.0225),1.7e-07,8.7e-07,significantly better


In [9]:
feat.drop(columns=["split", "TARGET"]).to_parquet(HERE / "features.parquet", index=False)
history_manifest = {
    "bureau": BUREAU + NEW_BUREAU + NEW_BB,
    "previous": PREVIOUS + NEW_PREV,
    "repayment": REPAYMENT + NEW_POS + NEW_CARD + NEW_INST,
}
assert sum(map(len, history_manifest.values())) == 31
update_metadata({"history_features": history_manifest})
rich_predictions["All sources"].to_csv(HERE / "best_model_scores.csv", index=False)
print("Saved features.parquet, best_model_scores.csv and feature definitions in model_metadata.json")


## 5. Feature selection on the 31-history-feature specification

Compare the full feature set (151 raw candidate columns, normally 281 encoded inputs) with a reduced set. The full set includes loan-level aggregates and monthly bureau delinquency summaries. Reuse the same fixed model settings.

1. Use **training applicants only** to identify columns missing more than 95% of values or having one value.
2. Find numeric pairs with absolute Spearman correlation above 0.9 on train; retain the feature with higher univariate train AUC.
3. Refit full and selected models and evaluate them with the shared evaluator. Do not assume selection improves performance.

In [10]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Reuse PROJECT_ROOT discovered in Section 0.
CLEAN = PROJECT_ROOT / "clean_data"
# Reuse HERE = PROJECT_ROOT / "Exp_1" / "exp1_outputs" from Section 0.
THR = 0.9          # correlation threshold
MISS_MAX = 0.95    # drop features missing for more than this share of applicants

pass  # feat comes from Section 4
splits = pd.read_csv(CLEAN / "splits.csv")[["SK_ID_CURR", "TARGET", "split"]]
pass  # already has split and TARGET
tr = feat["split"] == "train"
y_tr = feat.loc[tr, "TARGET"]

EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]
candidates = [c for c in feat.columns if c not in EXCLUDE]
numeric = [c for c in candidates if pd.api.types.is_numeric_dtype(feat[c])]
categorical = [c for c in candidates if c not in numeric]
print(f"{len(candidates)} candidate features: {len(numeric)} numeric, {len(categorical)} categorical")

In [11]:
miss = feat.loc[tr, candidates].isna().mean().sort_values(ascending=False)
print("Most missing features (share of training applicants):")
display(miss.head(10).round(3).to_frame("missing_share"))

drop_missing = miss[miss > MISS_MAX].index.tolist()
nunique = feat.loc[tr, candidates].nunique(dropna=True)
drop_constant = nunique[nunique <= 1].index.tolist()
print("Dropped for missingness:", drop_missing or "none")
print("Dropped as constant:", drop_constant or "none")

kept = [c for c in candidates if c not in drop_missing + drop_constant]
numeric = [c for c in numeric if c in kept]
categorical = [c for c in categorical if c in kept]

,missing_share
cc_min_ratio_mean,0.806
cc_balance_trend,0.786
cc_util_mean,0.719
cc_months_dpd,0.716
cc_months_below_min,0.716
cc_months,0.716
bb_max_dpd,0.710
bb_ever_late_share,0.702
bb_months_with_dpd,0.702
COMMONAREA_MODE,0.700


In [12]:
X_tr = feat.loc[tr, numeric]
corr = X_tr.corr(method="spearman").abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(ascending=False)
pairs = pairs[pairs > THR]
print(f"{len(pairs)} numeric pairs with |rho| > {THR}")
display(pairs.head(25).round(3).to_frame("abs_spearman"))

,,abs_spearman
FLAG_EMP_PHONE,DAYS_EMPLOYED_ANOM,1.000
YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998
YEARS_BEGINEXPLUATATION_AVG,YEARS_BEGINEXPLUATATION_MEDI,0.997
OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.997
LIVINGAPARTMENTS_AVG,LIVINGAPARTMENTS_MEDI,0.996
FLOORSMIN_AVG,FLOORSMIN_MEDI,0.996
COMMONAREA_AVG,COMMONAREA_MEDI,0.996
LANDAREA_AVG,LANDAREA_MEDI,0.996
LIVINGAREA_AVG,LIVINGAREA_MEDI,0.995
APARTMENTS_AVG,APARTMENTS_MEDI,0.995


In [13]:
def univariate_auc(col):
    x = X_tr[col]
    x = x.fillna(x.median())
    a = roc_auc_score(y_tr, x)
    return max(a, 1 - a)

auc = {c: univariate_auc(c) for c in numeric}

dropped, rows = set(), []
for (a, b), r in pairs.items():
    if a in dropped or b in dropped:
        continue
    weaker, stronger = (a, b) if auc[a] < auc[b] else (b, a)
    dropped.add(weaker)
    rows.append({"dropped": weaker, "kept": stronger, "abs_spearman": round(r, 3),
                 "AUC_dropped": round(auc[weaker], 3), "AUC_kept": round(auc[stronger], 3)})
choices = pd.DataFrame(rows)
display(choices)
selected = [c for c in kept if c not in dropped]
print(f"Dropped {len(dropped)} correlated features; {len(selected)} features selected in total")

,dropped,kept,abs_spearman,AUC_dropped,AUC_kept
0,DAYS_EMPLOYED_ANOM,FLAG_EMP_PHONE,1.000,0.536,0.536
1,YEARS_BUILD_MEDI,YEARS_BUILD_AVG,0.998,0.511,0.511
2,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BEGINEXPLUATATION_AVG,0.997,0.520,0.520
3,OBS_60_CNT_SOCIAL_CIRCLE,OBS_30_CNT_SOCIAL_CIRCLE,0.997,0.507,0.508
4,LIVINGAPARTMENTS_MEDI,LIVINGAPARTMENTS_AVG,0.996,0.508,0.508
5,FLOORSMIN_MEDI,FLOORSMIN_AVG,0.996,0.513,0.513
6,COMMONAREA_MEDI,COMMONAREA_AVG,0.996,0.505,0.505
7,LANDAREA_MEDI,LANDAREA_AVG,0.996,0.507,0.507
8,LIVINGAREA_MEDI,LIVINGAREA_AVG,0.995,0.526,0.527
9,APARTMENTS_AVG,APARTMENTS_MEDI,0.995,0.519,0.520


In [14]:
update_metadata({"selected_features": selected})
print(f"Recorded {len(selected)} selected features in model_metadata.json")


In [15]:
PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

full_cols = kept
sel_cols = selected
FILES = {}
for name, cols in [("All sources (full)", full_cols), ("All sources (selected)", sel_cols)]:
    X = model_inputs(cols)
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], y_tr)
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    FILES[name] = out
    print(f"{name:<24} columns={len(cols):>3} dummy inputs={X.shape[1]:>4} trees={model.n_iter_:>4}")
selection_predictions = {name: df.copy() for name, df in FILES.items()}


In [16]:
spec = importlib.util.spec_from_file_location("eval_exp1", PROJECT_ROOT / "Exp_1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(FILES)
table, comps, _ = evaluate_and_record(ev, [FILES[l] for l in labels], labels, n_boot=1000, stage="selection_31_history")
display(ev.pretty_metrics(table))
display(ev.pretty_comparisons(comps))

,threshold,pr_auc,roc_auc,precision,recall,f1
model,,,,,,
All sources (full),0.144,0.250 (0.228-0.273),0.758 (0.743-0.772),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321)
All sources (selected),0.146,0.249 (0.228-0.272),0.758 (0.743-0.772),0.233 (0.216-0.251),0.433 (0.406-0.461),0.303 (0.283-0.323)


,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,All sources (selected),All sources (full),pr_auc,-0.0012 (-0.0055 to +0.0025),0.58,1,no significant difference
1,All sources (selected),All sources (full),roc_auc,+0.0001 (-0.0022 to +0.0026),0.94,1,no significant difference


## 6. Final comparison: model family, historical features and feature selection

The exported tables contain nine unique models. Six configurations compare model families, feature selection and demographic inclusion. Three additional gradient-boosting configurations add Bureau, Previous applications or Repayment behaviour separately to application features. The shared application-only and all-sources models complete the five-group source ablation without duplicate rows.

Both all-sources model families use the **same 31 historical features** and exclude gender/marital status; age is retained. Logistic regression uses training-fitted median imputation and scaling. Its existing regularisation settings are reused (`C=1.0` for application-only, `C=0.1` for all-sources), without further test-driven tuning. The gradient-boosting settings remain fixed. Feature selection uses the training-derived list from Section 5.

Eight planned comparisons answer the questions directly: history within each model family, the separate bureau/previous-application/repayment contributions, model family on identical inputs, selected versus full features, and demographic inclusion versus exclusion. Holm correction is applied across all sixteen tests (two metrics per comparison).

Earlier trials remain in their notebook sections. The current five-group source ablation is included in the final CSVs: `exp1_metrics.csv` contains nine unique model rows and `exp1_comparisons.csv` contains sixteen paired-test rows. The application-only and all-sources configurations are reused across the main and supporting analyses. Predictions for intermediate models stay in memory. The existing investigation score file is unchanged.



**Source ablation:** Application only, Application plus Bureau, Application plus Previous applications, Application plus Repayment behaviour, and All sources use the same gradient-boosting settings. Their purpose is to explain which historical sources support the all-sources result. The source-ablation table and ROC/PR curves are displayed separately below the main model comparison.

### Saved source-contribution results

| Feature set | PR-AUC | ROC-AUC |
|---|---:|---:|
| GBM application | 0.21505 | 0.74101 |
| GBM application + Bureau | 0.22303 | 0.74306 |
| GBM application + Previous | 0.22177 | 0.74291 |
| GBM application + Repayment | 0.24219 | 0.75653 |
| GBM all sources | 0.25012 | 0.75754 |

Repayment behaviour produces the largest observed individual gain. Repayment and All sources improve both metrics significantly after the combined sixteen-test Holm correction; Bureau and Previous alone do not. These tests compare each source addition with Application only, not directly with one another.


In [ ]:
def build_final_models(project, include_full=True):
    """Model family, five-source ablation, feature selection and demographics."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    output = project / "Exp_1/exp1_outputs"
    demographic = pd.read_csv(output / "sensitivity_scores.csv")
    best = pd.read_csv(output / "best_model_scores.csv")
    with_attributes = demographic[["SK_ID_CURR", "score_with_attributes"]].rename(
        columns={"score_with_attributes": "score"})
    saved = {"GBM all sources": best, "GBM all sources + demographics": with_attributes}
    if not include_full:
        return saved

    metadata = json.loads((output / "model_metadata.json").read_text())
    specification = metadata["sensitivity"]
    all_columns = specification["feature_sets"]["Without gender and marital status"]
    history = {c for cols in metadata["history_features"].values() for c in cols}
    application = [c for c in all_columns if c not in history]
    selected = metadata["selected_features"]
    assert len(history) == 31 and len(application) == 120
    assert set(selected).issubset(all_columns)
    features = pd.read_parquet(output / "features.parquet")
    split = pd.read_csv(project / "clean_data/splits.csv")[["SK_ID_CURR", "TARGET", "split"]]
    features = features.merge(split, on="SK_ID_CURR", validate="one_to_one")
    train = features["split"].eq("train")
    scored = features["split"].isin(["val", "test"])
    # Reuse the existing LR regularisation settings; do not tune on the test set.
    models = {
        "LR application": (application, make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(C=1.0, max_iter=2000))),
        "LR all sources": (all_columns, make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(C=0.1, max_iter=2000))),
        "GBM application": (application, HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
        "GBM application + Bureau": (application + metadata["history_features"]["bureau"], HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
        "GBM application + Previous": (application + metadata["history_features"]["previous"], HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
        "GBM application + Repayment": (application + metadata["history_features"]["repayment"], HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
        "GBM selected features": (selected, HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
    }
    predictions = {}
    for label, (columns, model) in models.items():
        X = pd.get_dummies(features[columns], dummy_na=True).astype("float32")
        model.fit(X.loc[train], features.loc[train, "TARGET"])
        predictions[label] = features.loc[scored, ["SK_ID_CURR"]].assign(
            score=model.predict_proba(X.loc[scored])[:, 1])
        print(f"Trained {label}: {X.shape[1]} inputs; predictions kept in memory")
    return {
        "LR application": predictions["LR application"],
        "LR all sources": predictions["LR all sources"],
        "GBM application": predictions["GBM application"],
        "GBM application + Bureau": predictions["GBM application + Bureau"],
        "GBM application + Previous": predictions["GBM application + Previous"],
        "GBM application + Repayment": predictions["GBM application + Repayment"],
        "GBM all sources": best,
        "GBM selected features": predictions["GBM selected features"],
        "GBM all sources + demographics": with_attributes,
    }

def evaluate_final_models(engine, models, n_boot=1000):
    """Evaluate nine unique models and eight planned contrasts with the shared engine."""
    rows, evaluated = [], {}
    test_ids = set(engine.load_splits().query("split == 'test'")["SK_ID_CURR"])
    val_ids = set(engine.load_splits().query("split == 'val'")["SK_ID_CURR"])
    for label, values in models.items():
        frame = engine.load_predictions(values)
        if frame["SK_ID_CURR"].duplicated().any() or not np.isfinite(frame["score"]).all():
            raise ValueError(f"{label}: invalid or duplicate scores")
        val = frame[frame["split"].eq("val")].sort_values("SK_ID_CURR")
        test = frame[frame["split"].eq("test")].sort_values("SK_ID_CURR")
        if set(test["SK_ID_CURR"]) != test_ids or set(val["SK_ID_CURR"]) != val_ids:
            raise ValueError(f"{label}: must score every shared validation and test applicant")
        result = engine.evaluate(test.TARGET, test.score, y_val=val.TARGET, s_val=val.score, n_boot=n_boot)
        row = {"model": label, "threshold": result.attrs["threshold"]}
        for metric, values in result.iterrows():
            row[metric] = values["value"]
            row[metric + "_ci_low"] = values["ci_low"]
            row[metric + "_ci_high"] = values["ci_high"]
        if set(engine.SHOW) - set(result.index):
            raise ValueError("The final evaluator omitted a metric from the shared engine.")
        rows.append(row)
        evaluated[label] = {"val": val, "test": test, "threshold": result.attrs["threshold"]}
    pairs = [
        ("LR all sources", "LR application"),
        ("GBM all sources", "GBM application"),
        ("GBM application + Bureau", "GBM application"),
        ("GBM application + Previous", "GBM application"),
        ("GBM application + Repayment", "GBM application"),
        ("GBM all sources", "LR all sources"),
        ("GBM selected features", "GBM all sources"),
        ("GBM all sources + demographics", "GBM all sources"),
    ]
    comparisons = []
    for label, baseline in pairs:
        if label not in evaluated or baseline not in evaluated:
            continue
        trial, base = evaluated[label]["test"], evaluated[baseline]["test"]
        assert np.array_equal(trial.SK_ID_CURR, base.SK_ID_CURR)
        for metric in ("pr_auc", "roc_auc"):
            result = engine.compare_models(base.TARGET, trial.score, base.score, metric, n_boot)
            comparisons.append({"model": label, "vs_baseline": baseline, "metric": metric,
                "delta": result["delta"], "ci_low": result["ci_low"], "ci_high": result["ci_high"],
                "p_bootstrap": result["p_bootstrap"],
                "p_delong": result["delong_roc_auc"]["p_value"] if metric == "roc_auc" else np.nan})
    # Adjust across the complete set of planned tests, rather than separately by section.
    return pd.DataFrame(rows).set_index("model"), engine._add_p_values(pd.DataFrame(comparisons)), evaluated


QUEUE_METRICS = ["precision@top5%", "recall@top5%", "precision@top10%", "recall@top10%"]


def add_review_queue_metrics(table, evaluated, n_boot=1000, seed=42):
    """Supplement shared metrics with capacity-based review queues on test applicants."""
    result = table.copy()
    for label, prediction in evaluated.items():
        test = prediction["test"].sort_values("SK_ID_CURR")
        y = test["TARGET"].to_numpy(dtype=int)
        score = test["score"].to_numpy(dtype=float)
        ids = test["SK_ID_CURR"].to_numpy()

        def queue_values(index):
            # Higher scores first; applicant ID breaks score ties deterministically.
            order = np.lexsort((ids[index], -score[index]))
            ordered_y = y[index][order]
            positives = int(ordered_y.sum())
            values = {}
            for fraction, suffix in [(0.05, "5%"), (0.10, "10%")]:
                k = max(1, int(np.ceil(len(index) * fraction)))
                caught = int(ordered_y[:k].sum())
                values["precision@top" + suffix] = caught / k
                values["recall@top" + suffix] = caught / positives if positives else np.nan
            return values

        point = queue_values(np.arange(len(y)))
        rng = np.random.default_rng(seed)
        samples = []
        for _ in range(n_boot):
            index = rng.integers(0, len(y), len(y))
            if 0 < y[index].sum() < len(index):
                samples.append(queue_values(index))
        boot = pd.DataFrame(samples)
        for metric, value in point.items():
            result.loc[label, metric] = value
            result.loc[label, metric + "_ci_low"] = boot[metric].quantile(0.025)
            result.loc[label, metric + "_ci_high"] = boot[metric].quantile(0.975)
    return result


def display_ranking_curves(evaluated, labels=None):
    """Show ROC and precision-recall curves inline without extra output files."""
    import io
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from IPython.display import Image, display
    from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    # Four core configurations keep the main model-family comparison readable.
    labels = labels or [label for label in ["LR application", "LR all sources", "GBM application", "GBM all sources"]
              if label in evaluated]
    if not labels:
        labels = list(evaluated)
    for label in labels:
        test = evaluated[label]["test"]
        y, score = test["TARGET"], test["score"]
        fpr, tpr, _ = roc_curve(y, score)
        precision, recall, _ = precision_recall_curve(y, score)
        axes[0].plot(fpr, tpr, label=f"{label} (AUC {roc_auc_score(y, score):.3f})")
        axes[1].plot(recall, precision, label=f"{label} (AP {average_precision_score(y, score):.3f})")
    prevalence = evaluated[labels[0]]["test"]["TARGET"].mean()
    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random ranking")
    axes[1].axhline(prevalence, color="black", linestyle="--", linewidth=1,
                    label=f"Test default rate ({prevalence:.3f})")
    axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Test ROC curves")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Test precision-recall curves")
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=140)
    plt.close(fig)
    display(Image(data=buffer.getvalue()))



SOURCE_ABLATION_MODELS = [
    "GBM application", "GBM application + Bureau", "GBM application + Previous",
    "GBM application + Repayment", "GBM all sources",
]
MAIN_COMPARISON_MODELS = [
    "LR application", "LR all sources", "GBM application", "GBM all sources",
    "GBM selected features", "GBM all sources + demographics",
]


def display_source_ablation(engine, metrics, evaluated):
    labels = [label for label in SOURCE_ABLATION_MODELS if label in metrics.index]
    if len(labels) != 5:
        return
    print("Source contribution: identical gradient-boosting settings; application-only baseline")
    display(engine.pretty_metrics(metrics.loc[labels]))
    display(engine.pretty_metrics(metrics.loc[labels], metrics=QUEUE_METRICS))
    display_ranking_curves(evaluated, labels=labels)

final_models = build_final_models(PROJECT_ROOT)
final_metrics, final_comparisons, final_evaluated = evaluate_final_models(ev, final_models, n_boot=1000)
final_metrics = add_review_queue_metrics(final_metrics, final_evaluated, n_boot=1000)
final_metrics.to_csv(HERE / "exp1_metrics.csv")
final_comparisons.to_csv(HERE / "exp1_comparisons.csv", index=False)
display(ev.pretty_metrics(final_metrics.loc[MAIN_COMPARISON_MODELS]))
print("Supplementary review-queue metrics (95% bootstrap intervals):")
display(ev.pretty_metrics(final_metrics.loc[MAIN_COMPARISON_MODELS], metrics=QUEUE_METRICS))
display_ranking_curves(final_evaluated)
display_source_ablation(ev, final_metrics, final_evaluated)
display(ev.pretty_comparisons(final_comparisons))
print("Saved nine unique models, including the source ablation, and sixteen planned metric tests.")

## 7. Sensitivity analysis: gender and marital status

**Question:** How much does adding `CODE_GENDER` and `NAME_FAMILY_STATUS` change the performance and group error rates of the best all-sources gradient-boosting specification?

This is a paired comparison of **excluding both** versus **including both**. It does not isolate the effect of each attribute individually. Both versions retain age (`DAYS_BIRTH`), use the same 31 historical features, shared applicant splits, one-hot encoding, fixed hyperparameters and seed. There is no new hyperparameter search or feature selection. Both variants use identical loan-level aggregates and monthly delinquency summaries; the only added inputs are gender and marital status.

**Run independently:** run the following cells from this section in order; earlier sections need not be rerun. Features are rebuilt from the shared cleaned tables. The training-only category schema is applied to validation/test (unseen categories get zero dummy indicators). Numeric missing values remain NaN for histogram gradient boosting. Output files go to `Exp_1/exp1_outputs/`.

The existing Exp1 test results have already been inspected, so this is an **additional sensitivity analysis**, not a fresh untouched-test confirmation. Keep the excluded-attribute model as the main specification; do not select a replacement using these test results. These cells do not overwrite the scores supplied to Exp3.

In [18]:
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

sensitivity_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "Exp_1/exp1.ipynb").is_file()), None)
if sensitivity_root is None:
    raise FileNotFoundError("Cannot locate the project Exp_1 folder.")
sensitivity_output = sensitivity_root / "Exp_1/exp1_outputs"
sensitivity_output.mkdir(parents=True, exist_ok=True)
sensitivity_spec = importlib.util.spec_from_file_location(
    "eval_exp1_sensitivity", sensitivity_root / "Exp_1/eval_exp1.py")
sensitivity_ev = importlib.util.module_from_spec(sensitivity_spec)
sensitivity_spec.loader.exec_module(sensitivity_ev)

sensitivity_params = dict(learning_rate=0.03, max_leaf_nodes=15,
                          min_samples_leaf=100, l2_regularization=1.0)
sensitivity_fixed = dict(max_iter=1000, early_stopping=True,
                         validation_fraction=0.1, n_iter_no_change=20, random_state=42)


def write_result_rows(rows, section, record_type, folder=sensitivity_output):
    """Intermediate results are displayed only; final CSVs are written in Section 6."""
    return




def update_metadata(values, folder=sensitivity_output):
    target = Path(folder) / "model_metadata.json"
    existing = json.loads(target.read_text()) if target.exists() else {}
    existing.update(values)
    target.write_text(json.dumps(existing, indent=2))


def evaluate_and_record(engine, predictions, labels=None, *, stage, folder=sensitivity_output, **kwargs):
    metrics, comparisons, evaluated = engine.compare_prediction_files(predictions, labels, **kwargs)
    write_result_rows(metrics, stage, "test_model", folder)
    write_result_rows(comparisons, stage, "paired_comparison", folder)
    return metrics, comparisons, evaluated

# Local names keep the earlier notebook's features, models and prediction paths intact.
def build_sensitivity_features(clean_dir):
    CLEAN = clean_dir
    splits = pd.read_csv(CLEAN / "splits.csv")
    EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]
    EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]

    app = pd.read_parquet(CLEAN / "application.parquet").drop(columns=["TARGET"])
    app = app.merge(splits[["SK_ID_CURR", "TARGET", "split"]], on="SK_ID_CURR", how="inner")
    app_cols = [c for c in app.columns if c not in EXCLUDE]
    print(f"{len(app):,} applicants, {len(app_cols)} application features")

    # Bureau: loans at other lenders
    b = pd.read_parquet(CLEAN / "bureau.parquet")
    bureau = b.groupby("SK_ID_CURR").agg(
        n_bureau=("SK_ID_BUREAU", "size"),
        bureau_debt=("AMT_CREDIT_SUM_DEBT", "sum"),
        bureau_overdue=("AMT_CREDIT_SUM_OVERDUE", "sum"),
        bureau_credit=("AMT_CREDIT_SUM", "sum"),
    )
    bureau["bureau_debt_ratio"] = bureau["bureau_debt"] / bureau["bureau_credit"].where(bureau["bureau_credit"] > 0)

    # Previous Home Credit applications (duplicate submissions removed)
    p = pd.read_parquet(CLEAN / "previous_application.parquet")
    p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
    p["refused"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Refused").astype(int)
    prev = p.groupby("SK_ID_CURR").agg(
        n_prev=("SK_ID_PREV", "size"),
        prev_refusal_rate=("refused", "mean"),
        prev_amt_mean=("AMT_APPLICATION", "mean"),
    )

    # Instalments: one row per instalment (split payments already merged in cleaning)
    i = pd.read_parquet(CLEAN / "installments.parquet")
    paid = i["DAYS_ENTRY_PAYMENT"].notna()
    i["late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]) > 0, np.nan)   # unpaid: not counted
    i["under"] = (i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)).clip(lower=0)
    inst = i.groupby("SK_ID_CURR").agg(
        n_inst=("SK_ID_PREV", "size"),
        late_rate=("late", "mean"),
        underpay_mean=("under", "mean"),
    )

    # Credit cards: utilisation
    c = pd.read_parquet(CLEAN / "credit_card.parquet")
    c["util"] = c["AMT_BALANCE"] / c["AMT_CREDIT_LIMIT_ACTUAL"].where(c["AMT_CREDIT_LIMIT_ACTUAL"] > 0)
    cc = c.groupby("SK_ID_CURR").agg(cc_util_mean=("util", "mean"), cc_months=("MONTHS_BALANCE", "nunique"))

    hist = bureau.join([prev, inst, cc], how="outer").reset_index()
    hist_cols = [col for col in hist.columns if col != "SK_ID_CURR"]
    print(f"History features: {len(hist_cols)} columns for {len(hist):,} applicants with any history")

    # POS / cash loans
    q = pd.read_parquet(CLEAN / "pos_cash.parquet", columns=["SK_ID_CURR", "SK_DPD", "CNT_INSTALMENT_FUTURE"])
    q["dpd_flag"] = (q["SK_DPD"] > 0).astype("int32")
    pos = q.groupby("SK_ID_CURR").agg(
        pos_max_dpd=("SK_DPD", "max"),
        pos_months_dpd=("dpd_flag", "sum"),
        pos_remaining_inst=("CNT_INSTALMENT_FUTURE", "mean"),
    )

    # Credit card, month by month (one row per applicant-month, summed over contracts)
    c = pd.read_parquet(CLEAN / "credit_card.parquet", columns=[
        "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
        "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_TOTAL_CURRENT", "SK_DPD"])
    c["below_min"] = ((c["AMT_INST_MIN_REGULARITY"] > 0) &
                      (c["AMT_PAYMENT_TOTAL_CURRENT"].fillna(0) < c["AMT_INST_MIN_REGULARITY"])).astype("int32")
    c["dpd_flag"] = (c["SK_DPD"] > 0).astype("int32")
    c["min_ratio"] = c["AMT_PAYMENT_TOTAL_CURRENT"] / c["AMT_INST_MIN_REGULARITY"].where(c["AMT_INST_MIN_REGULARITY"] > 0)
    m = c.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False).agg(
        bal=("AMT_BALANCE", "sum"), lim=("AMT_CREDIT_LIMIT_ACTUAL", "sum"),
        below=("below_min", "max"), dpd=("dpd_flag", "max"), min_ratio=("min_ratio", "mean"))
    m["util"] = m["bal"] / m["lim"].where(m["lim"] > 0)
    recent = m[m["MONTHS_BALANCE"] >= -3].groupby("SK_ID_CURR")["util"].mean()
    card = m.groupby("SK_ID_CURR").agg(
        cc_months_below_min=("below", "sum"),
        cc_months_dpd=("dpd", "sum"),
        cc_min_ratio_mean=("min_ratio", "mean"),
        cc_util_overall=("util", "mean"),
    )
    card["cc_balance_trend"] = recent.reindex(card.index) - card["cc_util_overall"]
    card = card.drop(columns="cc_util_overall")

    # Instalments
    i = pd.read_parquet(CLEAN / "installments.parquet", columns=[
        "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"])
    paid = i["DAYS_ENTRY_PAYMENT"].notna()
    i["days_late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]).clip(lower=0), np.nan)
    i["under_flag"] = ((i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)) > 1.0).astype("int32")
    i["paid_amt"] = i["AMT_PAYMENT"].fillna(0)
    inst_new = i.groupby("SK_ID_CURR").agg(
        inst_max_days_late=("days_late", "max"),
        inst_mean_days_late=("days_late", "mean"),
        inst_underpay_freq=("under_flag", "mean"),
        paid_sum=("paid_amt", "sum"),
        due_sum=("AMT_INSTALMENT", "sum"),
    )
    inst_new["inst_paid_to_due"] = inst_new["paid_sum"] / inst_new["due_sum"].where(inst_new["due_sum"] > 0)
    inst_new = inst_new.drop(columns=["paid_sum", "due_sum"])

    # Bureau (loan-level)
    b = pd.read_parquet(CLEAN / "bureau.parquet", columns=["SK_ID_CURR", "CREDIT_DAY_OVERDUE", "CREDIT_ACTIVE"])
    bureau_new = b.groupby("SK_ID_CURR").agg(
        bureau_max_days_overdue=("CREDIT_DAY_OVERDUE", "max"),
        bureau_active_count=("CREDIT_ACTIVE", lambda s: (s.astype(str) == "Active").sum()),
    )

    # Bureau-balance summaries of monthly delinquency at external lenders
    bb_flat = pd.read_parquet(CLEAN / "bureau_balance.parquet", columns=["SK_ID_CURR", "STATUS_DPD"])
    bb_flat["dpd_flag"] = (bb_flat["STATUS_DPD"].fillna(0) > 0).astype("int32")
    bb_agg = bb_flat.groupby("SK_ID_CURR").agg(
        bb_max_dpd=("STATUS_DPD", "max"),
        bb_months_with_dpd=("dpd_flag", "sum"),
        bb_ever_late_share=("dpd_flag", "mean"),
    )
    print(f"Bureau balance flat: {len(bb_agg):,} applicants")

    # Previous applications
    p = pd.read_parquet(CLEAN / "previous_application.parquet", columns=[
        "SK_ID_CURR", "NAME_CONTRACT_STATUS", "DAYS_DECISION", "IS_DUPLICATE_SUBMISSION"])
    p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
    p["approved"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Approved").astype("int32")
    p["last_year"] = (p["DAYS_DECISION"] >= -365).astype("int32")
    prev_new = p.groupby("SK_ID_CURR").agg(prev_approval_rate=("approved", "mean"), prev_apps_last_year=("last_year", "sum"))

    new_hist = pos.join([card, inst_new, bureau_new, bb_agg, prev_new], how="outer").reset_index()
    NEW_POS = ["pos_max_dpd", "pos_months_dpd", "pos_remaining_inst"]
    NEW_CARD = ["cc_months_below_min", "cc_months_dpd", "cc_min_ratio_mean", "cc_balance_trend"]
    NEW_INST = ["inst_max_days_late", "inst_mean_days_late", "inst_underpay_freq", "inst_paid_to_due"]
    NEW_BUREAU = ["bureau_max_days_overdue", "bureau_active_count"]
    NEW_BB = ["bb_max_dpd", "bb_months_with_dpd", "bb_ever_late_share"]
    NEW_PREV = ["prev_approval_rate", "prev_apps_last_year"]
    assert set(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV) == set(new_hist.columns) - {"SK_ID_CURR"}
    print(f"New features: {len(new_hist.columns) - 1} columns for {len(new_hist):,} applicants")
    display(new_hist.describe().T[["count", "mean", "min", "max"]].round(3))
    frame = app.merge(hist, on="SK_ID_CURR", how="left", validate="one_to_one")
    frame = frame.merge(new_hist, on="SK_ID_CURR", how="left", validate="one_to_one")
    columns = app_cols + hist_cols + NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV
    return frame, columns

sensitivity_data, sensitivity_base_cols = build_sensitivity_features(sensitivity_root / "clean_data")
assert sensitivity_data["SK_ID_CURR"].is_unique
assert len(sensitivity_base_cols) == 151
assert "DAYS_BIRTH" in sensitivity_base_cols
assert not {"CODE_GENDER", "NAME_FAMILY_STATUS"}.intersection(sensitivity_base_cols)
sensitivity_sets = {
    "Without gender and marital status": sensitivity_base_cols,
    "With gender and marital status": sensitivity_base_cols + ["CODE_GENDER", "NAME_FAMILY_STATUS"],
}
print("Shared data:", sensitivity_shared)
print("Outputs:", sensitivity_output)

,count,mean,min,max
SK_ID_CURR,99290.0,277928.762,100011.00,456248.000
pos_max_dpd,94164.0,15.262,0.00,4231.000
pos_months_dpd,94164.0,0.885,0.00,120.000
pos_remaining_inst,94155.0,9.215,0.00,59.500
cc_months_below_min,28368.0,2.096,0.00,70.000
cc_months_dpd,28368.0,1.546,0.00,94.000
cc_min_ratio_mean,19351.0,34.067,0.00,471288.531
cc_balance_trend,21320.0,0.007,-0.97,2.767
inst_max_days_late,94873.0,18.639,0.00,2852.000
inst_mean_days_late,94873.0,1.282,0.00,2813.000


In [19]:
sensitivity_train = sensitivity_data["split"].eq("train")
sensitivity_val = sensitivity_data["split"].eq("val")
sensitivity_scored = sensitivity_data["split"].isin(["val", "test"])
sensitivity_files, sensitivity_validation_rows = {}, []
sensitivity_encoded_columns = {}

for sensitivity_label, sensitivity_columns in sensitivity_sets.items():
    # Fit the category vocabulary on train only and preserve its column order.
    sensitivity_Xtrain = pd.get_dummies(
        sensitivity_data.loc[sensitivity_train, sensitivity_columns], dummy_na=True).astype("float32")
    sensitivity_Xscore = pd.get_dummies(
        sensitivity_data.loc[sensitivity_scored, sensitivity_columns], dummy_na=True
    ).reindex(columns=sensitivity_Xtrain.columns, fill_value=0).astype("float32")
    sensitivity_model = HistGradientBoostingClassifier(**sensitivity_params, **sensitivity_fixed)
    sensitivity_model.fit(sensitivity_Xtrain, sensitivity_data.loc[sensitivity_train, "TARGET"])
    sensitivity_predictions = sensitivity_data.loc[sensitivity_scored, ["SK_ID_CURR"]].copy()
    sensitivity_predictions["score"] = sensitivity_model.predict_proba(sensitivity_Xscore)[:, 1]
    sensitivity_slug = "with_attributes" if sensitivity_label.startswith("With ") else "without_attributes"
    sensitivity_files[sensitivity_label] = sensitivity_predictions.copy()
    sensitivity_encoded_columns[sensitivity_label] = sensitivity_Xtrain.columns.tolist()
    sensitivity_val_scores = sensitivity_predictions.loc[
        sensitivity_data.loc[sensitivity_scored, "split"].eq("val"), "score"]
    sensitivity_validation_rows.append({
        "model": sensitivity_label,
        "raw_features": len(sensitivity_columns),
        "encoded_inputs": sensitivity_Xtrain.shape[1],
        "trees": sensitivity_model.n_iter_,
        "val_pr_auc": average_precision_score(sensitivity_data.loc[sensitivity_val, "TARGET"], sensitivity_val_scores),
        "val_roc_auc": roc_auc_score(sensitivity_data.loc[sensitivity_val, "TARGET"], sensitivity_val_scores),
    })

sensitivity_validation = pd.DataFrame(sensitivity_validation_rows).set_index("model")
write_result_rows(sensitivity_validation, "sensitivity_31_history", "validation_model", sensitivity_output)
display(sensitivity_validation)
update_metadata({"sensitivity": {
    "parameters": sensitivity_params, "fixed_settings": sensitivity_fixed,
    "feature_sets": sensitivity_sets, "encoded_columns": sensitivity_encoded_columns,
    "age_retained": True, "category_schema": "training data only; unknown categories -> all zeros",
    "sklearn_version": sklearn.__version__,
}}, sensitivity_output)
sensitivity_score_export = sensitivity_files["Without gender and marital status"].rename(columns={"score": "score_without_attributes"}).merge(
    sensitivity_files["With gender and marital status"].rename(columns={"score": "score_with_attributes"}),
    on="SK_ID_CURR", validate="one_to_one")
sensitivity_score_export.to_csv(sensitivity_output / "sensitivity_scores.csv", index=False)


,raw_features,encoded_inputs,trees,val_pr_auc,val_roc_auc
model,,,,,
Without gender and marital status,151,281,538,0.235911,0.758801
With gender and marital status,153,290,374,0.242051,0.763019


### 7.1 Paired test results

PR-AUC is primary; ROC-AUC is secondary. The shared evaluator chooses each model's operating threshold by maximum validation F1, then evaluates the same test applicants with 1,000 bootstrap resamples. The paired differences are **with attributes minus without attributes**. Use the Holm-adjusted p-values (`p_holm`) for significance; a confidence interval spanning zero indicates uncertainty. Threshold-dependent precision/recall can differ because thresholds differ, so also inspect ranking metrics and the common top-10% review budget.

In [20]:
sensitivity_labels = list(sensitivity_sets)
sensitivity_metrics, sensitivity_comparisons, sensitivity_evaluated = evaluate_and_record(sensitivity_ev, 
    [sensitivity_files[label] for label in sensitivity_labels], labels=sensitivity_labels,
    stage="sensitivity_31_history", folder=sensitivity_output, n_boot=1000, clean_dir=sensitivity_root / "clean_data")
display(sensitivity_ev.pretty_metrics(sensitivity_metrics))
display(sensitivity_ev.pretty_comparisons(sensitivity_comparisons))

# Fixed review capacity: top 10% of the same test applicants for each model.
sensitivity_queue_rows = []
for sensitivity_label in sensitivity_labels:
    sensitivity_test = sensitivity_evaluated[sensitivity_label]["test"]
    sensitivity_queue = sensitivity_test.sort_values(
        ["score", "SK_ID_CURR"], ascending=[False, True]).head(max(1, int(np.ceil(len(sensitivity_test) * 0.10))))
    sensitivity_positives = int(sensitivity_test["TARGET"].sum())
    sensitivity_queue_rows.append({
        "model": sensitivity_label, "reviewed": len(sensitivity_queue),
        "precision_at_top10pct": sensitivity_queue["TARGET"].mean(),
        "recall_at_top10pct": sensitivity_queue["TARGET"].sum() / sensitivity_positives if sensitivity_positives else np.nan,
    })
sensitivity_queue_metrics = pd.DataFrame(sensitivity_queue_rows).set_index("model")
write_result_rows(sensitivity_queue_metrics, "sensitivity_31_history", "review_queue", sensitivity_output)
display(sensitivity_queue_metrics)

,threshold,pr_auc,roc_auc,precision,recall,f1
model,,,,,,
Without gender and marital status,0.144,0.250 (0.228-0.273),0.758 (0.743-0.772),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321)
With gender and marital status,0.126,0.248 (0.226-0.271),0.759 (0.744-0.772),0.217 (0.203-0.232),0.500 (0.472-0.528),0.302 (0.285-0.320)


,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,With gender and marital status,Without gender and marital status,pr_auc,-0.0023 (-0.0065 to +0.0026),0.47,0.76,no significant difference
1,With gender and marital status,Without gender and marital status,roc_auc,+0.0012 (-0.0015 to +0.0041),0.38,0.76,no significant difference


,reviewed,precision_at_top10pct,recall_at_top10pct
model,,,
Without gender and marital status,1500,0.270667,0.33526
With gender and marital status,1500,0.270667,0.33526


### 7.2 Descriptive group error rates

Keep demographic fields available for auditing even when they are excluded as model inputs. Below, each model uses its **single validation-selected threshold** for all groups. FPR is the share of non-defaulters incorrectly flagged; FNR is the share of defaulters missed. Group PR-AUC depends on the group's default rate. Missing groups are shown explicitly; rates with zero denominators are NaN. Small groups have substantial uncertainty: these descriptive differences do not establish discrimination or fairness, and including/excluding attributes alone does not guarantee fairness. Age is retained in both models and is not tested by this two-attribute comparison.

In [21]:
sensitivity_group_rows = []
for sensitivity_label in sensitivity_labels:
    sensitivity_test = sensitivity_evaluated[sensitivity_label]["test"].merge(
        sensitivity_data[["SK_ID_CURR", "CODE_GENDER", "NAME_FAMILY_STATUS"]],
        on="SK_ID_CURR", how="left", validate="one_to_one")
    sensitivity_threshold = sensitivity_evaluated[sensitivity_label]["threshold"]
    sensitivity_test["predicted_high"] = sensitivity_test["score"].ge(sensitivity_threshold)
    for sensitivity_attribute in ["CODE_GENDER", "NAME_FAMILY_STATUS"]:
        sensitivity_groups = sensitivity_test[sensitivity_attribute].astype("string").fillna("<missing>")
        for sensitivity_group, sensitivity_part in sensitivity_test.groupby(sensitivity_groups, sort=True):
            sensitivity_y = sensitivity_part["TARGET"].eq(1)
            sensitivity_high = sensitivity_part["predicted_high"]
            sensitivity_npos = int(sensitivity_y.sum())
            sensitivity_nneg = int((~sensitivity_y).sum())
            sensitivity_group_rows.append({
                "model": sensitivity_label, "attribute": sensitivity_attribute, "group": sensitivity_group,
                "n": len(sensitivity_part), "defaulters": sensitivity_npos, "nondefaulters": sensitivity_nneg,
                "default_rate": sensitivity_y.mean(), "threshold": sensitivity_threshold,
                "flagged_share": sensitivity_high.mean(),
                "false_positive_rate": int((sensitivity_high & ~sensitivity_y).sum()) / sensitivity_nneg if sensitivity_nneg else np.nan,
                "false_negative_rate": int((~sensitivity_high & sensitivity_y).sum()) / sensitivity_npos if sensitivity_npos else np.nan,
                "pr_auc": average_precision_score(sensitivity_y, sensitivity_part["score"]) if sensitivity_npos and sensitivity_nneg else np.nan,
                "roc_auc": roc_auc_score(sensitivity_y, sensitivity_part["score"]) if sensitivity_npos and sensitivity_nneg else np.nan,
            })
sensitivity_group_metrics = pd.DataFrame(sensitivity_group_rows)
write_result_rows(sensitivity_group_metrics, "sensitivity_31_history", "group_errors", sensitivity_output)
display(sensitivity_group_metrics)
print("Sensitivity analysis saved to:", sensitivity_output)

,model,attribute,group,n,defaulters,nondefaulters,default_rate,threshold,flagged_share,false_positive_rate,false_negative_rate,pr_auc,roc_auc
0,Without gender and marital status,CODE_GENDER,F,9916,701,9215,0.070694,0.144494,0.135337,0.114487,0.590585,0.233440,0.752422
1,Without gender and marital status,CODE_GENDER,M,5084,510,4574,0.100315,0.144494,0.188434,0.156318,0.523529,0.276617,0.756907
2,Without gender and marital status,NAME_FAMILY_STATUS,Civil marriage,1397,143,1254,0.102362,0.144494,0.204724,0.169856,0.489510,0.309261,0.772722
3,Without gender and marital status,NAME_FAMILY_STATUS,Married,9573,725,8848,0.075734,0.144494,0.139977,0.119123,0.605517,0.222866,0.749107
4,Without gender and marital status,NAME_FAMILY_STATUS,Separated,909,62,847,0.068207,0.144494,0.126513,0.103896,0.564516,0.279523,0.775888
5,Without gender and marital status,NAME_FAMILY_STATUS,Single / not married,2339,242,2097,0.103463,0.144494,0.209491,0.172628,0.471074,0.312902,0.759205
6,Without gender and marital status,NAME_FAMILY_STATUS,Widow,782,39,743,0.049872,0.144494,0.088235,0.071332,0.589744,0.188214,0.738655
7,With gender and marital status,CODE_GENDER,F,9916,701,9215,0.070694,0.126256,0.155103,0.133370,0.559201,0.234876,0.754873
8,With gender and marital status,CODE_GENDER,M,5084,510,4574,0.100315,0.126256,0.246459,0.209226,0.419608,0.268975,0.753216
9,With gender and marital status,NAME_FAMILY_STATUS,Civil marriage,1397,143,1254,0.102362,0.126256,0.251253,0.214514,0.426573,0.302469,0.766370


### 7.3 What to report

Report both models' test PR-AUC and ROC-AUC with confidence intervals, the paired differences with Holm-adjusted p-values, and precision/recall at the common top-10% review budget. Describe group error rates with their denominators; avoid strong claims about small groups. A non-significant difference does not establish equivalence. If predictive performance improves when adding the attributes, report that finding alongside group errors rather than treating the improvement as proof that inclusion is appropriate. This comparison changes both attributes together and cannot attribute a change to just one.

The main Exp1 model remains the version excluding gender and marital status. This sensitivity analysis uses the full 31-feature historical representation and does not change the existing investigation cohort. The new section uses a training-only category schema, whereas the earlier notebook builds dummy columns on the full sample; both variants here use the same preprocessing, but exact agreement with previously exported scores is not guaranteed across schemas or library versions.

### 7.4 Saved results: 31 historical features

| Model | Test PR-AUC | Test ROC-AUC |
|---|---|---|
| Without gender and marital status | 0.25012 | 0.75754 |
| With gender and marital status | 0.24778 | 0.75875 |

Differences (with minus without): PR-AUC -0.00233, ROC-AUC +0.00121. Both paired confidence intervals include zero; both Holm-adjusted p-values are 0.764. Including these fields does not establish a predictive improvement in this run.

**Feature counts:** both variants use the same 31 historical features. The excluded-attribute model has 281 encoded inputs; adding gender and marital status increases this to 290. The comparison therefore isolates their joint inclusion while retaining age and the other model inputs. Each model uses early stopping under the same training rule.

The excluded model in this section reproduces Section 4's updated all-sources scores. `sensitivity_scores.csv` contains both attribute variants side by side. `exp1_metrics.csv` and `exp1_comparisons.csv` contain only the final model comparisons. Demographic group error rates remain displayed in this notebook. Detailed feature definitions and settings are in `model_metadata.json`. The existing `Exp_1/exp1_test_scores.csv` used by Exp3 is unchanged.
The adjusted p-values above refer to the two-test standalone sensitivity analysis. The final comparison in Section 6 corrects across all sixteen planned tests; both demographic-comparison adjusted p-values there are 1.000.